In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import hydra
import torch
from omegaconf import DictConfig

from genpp import BASE_DIR
from genpp.configs import register_resolvers

try:
    register_resolvers()
except Exception:
    print("Resolvers already registered:")

torch.set_float32_matmul_precision("high")

In [ ]:
with hydra.initialize_config_dir(config_dir=str(BASE_DIR / "configs"), version_base=None):
    cfg: DictConfig = hydra.compose(
        config_name="base_chen",
        overrides=["model/loss_fn=patchwise_rbf_score", "trainer=overfit"],
    )

In [ ]:
datamodule = hydra.utils.instantiate(cfg.data.module)
datamodule.prepare_data()
model = hydra.utils.instantiate(
    cfg.model, rescaler=datamodule.y_reverseModules if cfg.model.use_rescaler else None
)
model.compile(mode="max-autotune")
logger = False

trainer = hydra.utils.instantiate(cfg.trainer, logger=logger)

In [ ]:
trainer.fit(model, datamodule)

In [ ]:
datamodule.cleanup()